# Reproducibility, the Sign-Off, and the One-Page Brief

<hr>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb18_reproducibility_brief_student.ipynb)

---

## 🧭 Approach & Claim Boundary

**Approach emphasis:** all (reproducibility → communication) — the four
approaches are behind you now; these three meetings make what you found
*survivable*. First a stranger rebuilds your headline number from your package
alone; then you say what it all means on one page, to a reader who will never
open your appendix.

| | |
|---|---|
| **A claim this topic PERMITS** | "A competent stranger reproduced my headline number from my package alone — verified and dated — and every sentence of my one-page brief traces to a claim-ledger row." |
| **A claim this topic does NOT permit** | "It works on my machine" — reproducibility asserted but never exercised by another human — and a brief that quietly restores a caveat the ledger already disciplined. |

**Where this sits in the course:** meetings 39–41 of 44. It develops **M20 —
Reproducibility Audit** (due Wed Dec 2) and **M21 — Research Brief** (due Fri
Dec 4), and it stages the **M22 Evidence Defense**. It rests entirely on the
claim ledger you built at M19 — the spine everything from here forward is
traced back to.

*Provenance: RDSS ch. 21 'Planning' + ch. 22 'Realization' + ch. 23 'Integration' opening | planning/realization/integration | reproducibility-package protocol + one-page decision-maker brief for Colab-scale projects | extended; brief P8 (fresh)*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Assemble a **reproducibility package** a stranger could run: a
   restart-and-run-all notebook, a data-provenance statement, a seed +
   environment note, a decision log, and your finalized AI-use ledger.
2. Audit any package for the **five package sins** — hard-coded path, missing
   seed, by-hand edit, undocumented exclusion, stale data — and fix what breaks.
3. Run a **cold partner reproduction** and record an honest signed
   attestation — or a residuals log that is itself audit evidence.
4. Compress fourteen weeks into a **one-page research brief** (question,
   finding, uncertainty, implication, one honest figure) without un-verifying a
   single claim.
5. Trace every sentence of the brief to a ledger row, and **stage your Evidence
   Defense** by prepping your three weakest rows first.
6. Apply all of it to your own **M20 package** and **M21 brief**.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

## 1. Why This Matters

> *"Every retraction I have ever handled began as a result that could not be
> rebuilt. Not fraud — just a number nobody, including the author, could produce
> a second time. 'It works on my machine' is not a defense. It is the sound a
> paper makes right before it falls apart."*
> — a **journal reviewer**, on the quiet way good work dies

For fourteen weeks you have insisted that a claim is only as good as the
evidence someone can *check*. This week that standard turns on your own work.
Reproducibility is not a virtue you assert; it is a claim like any other, and
like any other it has to be **exercised by another human** — a competent
stranger who takes what you hand them and rebuilds your headline number, alone,
while you say nothing.

Three meetings, three artifacts, and then December is done: the **package**
(a stranger can run it), the **brief** (a decision-maker can act on it), and the
**defense** (you can stand behind it out loud). This notebook builds the first
two and stages the third.

> **A question that often comes up here:** *"My project is small — a pilot, a
> few hundred rows. Isn't a whole reproducibility package overkill?"* The size
> of the project is exactly why it is teachable, not why it is exempt. The
> habits are the point: a hard-coded path breaks a two-cell notebook as surely
> as a two-thousand-cell one, and the discipline you build on something small is
> the discipline you will actually have when the stakes are large. Small is
> where you learn to be trustworthy cheaply.

---

## 2. Anatomy of a Reproducibility Package

**Guiding question:** *what does a stranger need in order to rebuild your headline number without you in the room — and where do packages most reliably break?*

A package is **everything a stranger needs, and nothing they would have to
guess.** Five parts, each answering one question a reproducer will actually ask:

| Part | The question it answers | What "done" looks like |
|---|---|---|
| **Restart-and-run-all notebook** | Does it run, top to bottom, with no hand-holding? | One click from a fresh kernel produces every number — no manual steps, no cells run out of order. |
| **Data provenance statement** | Where did each input come from, and can I get the same one? | Source, version/date, and access conditions for every dataset. |
| **Seed + environment note** | Will I get the *same* numbers, not just *some* numbers? | Every random seed fixed in-code; the environment (Colab / package versions) recorded. |
| **Decision log** | Which by-hand choices shaped the result, and why? | Every exclusion, recode, cut point, and filter — each with its reason. |
| **Finalized AI-use ledger** | Which AI tool did what, and how was it checked? | The full-project record: tool, task (locate / draft / critique), verification — closed out. |

The engineering test that binds them together is **restart-and-run-all**: clear
the kernel, run every cell from the top, and see whether the headline number
comes back. If it only works when you click cells in the order you happen to
remember, you do not have a package — you have a performance that depends on you
being in the room.

**Where packages actually break** is boringly consistent, which is good news —
boring failures are catchable. Three culprits account for most of it: **paths**
(absolute paths that exist only on your laptop), **seeds** (a sample or split
that reshuffles every run), and **by-hand steps** (the one cell you edited once,
by hand, and forgot). Formalized, those and two siblings are **the five package
sins**:

1. **Hard-coded path** — `read_csv("/Users/you/Desktop/final.csv")` dies on
   every machine but yours.
2. **Missing seed** — any `sample`, split, or shuffle with no fixed seed gives a
   different answer each run.
3. **By-hand edit** — a value patched directly in a cell (or worse, in the file)
   that no clean run will ever reproduce.
4. **Undocumented exclusion** — rows dropped or filtered with no logged reason,
   so a reader cannot tell a decision from an accident.
5. **Stale / unversioned data** — an input with no source, date, or version — a
   reader cannot obtain the same one, so they cannot get the same number.

---

### 🔮 Pause & Predict

In a moment you will run a tiny auditor over a **deliberately sinful**
mini-package — five lines, each broken on purpose — and over a **clean**
counterpart of the same study.

**Before running the next cell**, commit a prediction in the cell below: how
many of the five sins do you think the auditor will flag in the sinful package,
and which single sin do you expect breaks a reproducer *first* when they hit the
Run button? One sentence on why.

### YOUR ANSWER HERE:

**Sins I expect the auditor to flag (0–5):**

**The sin I think breaks a reproducer first, and why:**

---

### 🛠️ Run the Study — the package self-audit

The cell below defines `audit_package`, a small, transparent checker. It does
not judge your ideas — only whether a stranger could *run* your work. Read each
check: it is exactly the mechanical thing a reproducer's eyes would catch. Then
watch it score a sinful package and a clean one. Later, you will paste your
**own** notebook's key lines into a list and run it on yourself.

In [ ]:
# A tiny reproducibility auditor. Each check is one thing a reproducer's eyes
# would catch. It scans a list of source LINES (strings) — paste your own below.

def audit_package(lines):
    text = "\n".join(lines).lower()
    sins = []

    # Sin 1 — a hard-coded absolute path: dies on every machine but its author's.
    if any(("/users/" in ln.lower()) or ("/home/" in ln.lower()) for ln in lines):
        sins.append("hard-coded path")

    # Sin 2 — no fixed seed anywhere: any sample/split/shuffle moves every run.
    if not any(k in text for k in ("seed", "random_state", "default_rng")):
        sins.append("missing seed")

    # Sin 3 — a value patched by hand: no clean run will reproduce it.
    if any((".loc[" in ln and "=" in ln) or ("by hand" in ln.lower())
           or ("manually" in ln.lower()) for ln in lines):
        sins.append("by-hand edit")

    # Sin 4 — an exclusion/filter with no logged reason nearby.
    filtered = [ln for ln in lines
                if (".drop(" in ln) or (".dropna(" in ln)
                or ("df[df[" in ln.replace(" ", ""))]
    if filtered and not any("reason" in ln.lower() for ln in lines):
        sins.append("undocumented exclusion")

    # Sin 5 — data with no shareable provenance: a reader can't reobtain the inputs.
    loads_csv = any("read_csv" in ln for ln in lines)
    shareable = any(("http" in ln.lower()) or ("load_course_data" in ln) for ln in lines)
    versioned = any(("version" in ln.lower()) or ("doi" in ln.lower()) for ln in lines)
    if loads_csv and not shareable and not versioned:
        sins.append("stale/unversioned data")

    return sins

In [ ]:
# A deliberately sinful mini-package — every line breaks reproducibility on purpose.
sinful = [
    'df = pd.read_csv("/Users/rivera/Desktop/thesis/office_hours_FINAL_v3.csv")',
    'sample = df.sample(200)                 # nothing fixes the shuffle',
    'df = df[df["year"] == 1]                # kept first-years only; no note on why',
    'df.loc[41, "visits"] = 3                # patched this row by hand post-survey',
    '# data pulled sometime last spring; nobody logged the download date',
]

# A clean counterpart — same study, nothing a stranger would have to guess.
clean = [
    "SEED = 464",
    "rng = np.random.default_rng(SEED)",
    'df = load_course_data("la_voter_file.csv")   # provenance + DOI logged in the ledger',
    'df = df[df["year"] == 1]   # reason: pre-registered to first-years; in the decision log',
    "sample = df.sample(200, random_state=SEED)",
]

print("Sinful package flags:", audit_package(sinful))
print("Clean  package flags:", audit_package(clean))

In [ ]:
# Self-check — the sinful package must trip all five checks, the clean one none.
assert audit_package(sinful) == ["hard-coded path", "missing seed", "by-hand edit",
                                 "undocumented exclusion", "stale/unversioned data"], \
    "auditor drifted — the sinful package should trip all five checks"
assert audit_package(clean) == [], "clean package should trip no checks"
print("✓ Self-check passed: 5 sins in the sinful package, 0 in the clean one.")

### 🔍 Reading the Evidence

Answer three things in the cell below. First: how did your prediction compare to
the auditor's five flags? Second — the important one — write the **most precise
claim a clean audit permits**. Careful: a package with zero flags is not a
package that is *correct*; it is a package that is *runnable*. Name the
difference. Third: name one reproducibility problem this auditor **cannot**
catch, no matter how many checks you add — something only a human reproducer,
running it cold, would find.

### YOUR ANSWER HERE:

**My prediction vs the five flags:**

**The most precise claim a clean audit permits (runnable ≠ correct):**

**One problem the auditor cannot catch, that a human reproducer would:**

---

The auditor is a warm-up for the real event: a human, running your work with
none of your memory of how it is supposed to go. First, drill the five sins by
eye — because in the exchange you will have to spot them without a script.

---

### 📝 Practice

Below is a five-line package. Each line commits **exactly one** of the five
sins. Match each line to its sin (hard-coded path / missing seed / by-hand edit
/ undocumented exclusion / stale-unversioned data):

- **A.** `scores = raw.sample(frac=0.3)`  *(and no seed is set anywhere in the notebook)*
- **B.** `df = pd.read_csv("C:/Users/kim/Documents/survey.csv")`
- **C.** `df = df[df["age"] < 90]`  *(no comment, no log entry)*
- **D.** `# spreadsheet downloaded at some point; not sure which export`
- **E.** `results.loc[7, "mean"] = 4.4   # typed in the corrected value myself`


### YOUR ANSWER HERE:

**A:** &nbsp; **B:** &nbsp; **C:** &nbsp; **D:** &nbsp; **E:**

---

### ⚖️ Make a Design Choice

Your own package, honestly, probably has one or two of these sins right now.
In the cell below: (1) name the sin you most suspect is hiding in your notebook;
(2) decide whether you will fix it **before** the exchange or let the exchange
find it — and defend the choice. There is a real argument each way (fixing first
saves a partner's time; letting it surface proves the exchange works), so make
the argument, do not just pick.

> 💡 **Gemini Prompt:** "Here is the description of my
> reproducibility package and the key cells of my notebook: [paste them]. Acting
> as a stranger who has never seen this project, list every place it might fail
> to reproduce on another machine — hard-coded paths, unset seeds, by-hand
> edits, undocumented exclusions, unversioned data — and rank them by how early
> each would break a cold run."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Confirm each alleged hole is real by looking at the actual cell — the
>   tool will invent plausible problems that are not in your code.
> - [ ] Fix only holes you verified; re-run **restart-and-run-all** yourself and
>   watch the headline number return before you believe the fix.
> - [ ] Log this use in your AI ledger: tool, task, how you verified.

### YOUR ANSWER HERE:

**The sin I most suspect is hiding in my package:**

**Fix it before the exchange, or let the exchange find it? My defense:**

---

### 🎯 Project Transfer — Exchange Round 1 (author silent)

This is the heart of M20. Swap packages with your partner. Your partner opens
**your** package cold and tries to reproduce your headline number from it
alone — and **you say nothing.** Not a hint, not a "oh, you have to run that one
first." Their confusion is your data. In the cell below, log the exchange as it
happens to you as the author:

Then do it in reverse — you reproduce theirs — and notice how much you reach for
context they never wrote down.

### YOUR ANSWER HERE:

**What my partner tried first, and where they got stuck:**

**The order things broke in (first breakage → second → …):**

**The one fix that would have prevented the most confusion:**

---

### 🎟️ Claim Ticket

**Claim Ticket #39** — in the cell below, name the **first thing that broke**
when a stranger ran your work, and your fix. One sentence. That sentence is your
ticket out the door today; tonight you fix everything the exchange exposed.

### YOUR ANSWER HERE:

**First breakage + my fix, one sentence:**

---

---

*(Meeting M40 picks up here.)*

## 3. The Sign-Off: Reproduction, Attested

> *"A signature is not a formality — it is a person putting their name on a
> claim. 'Reproduced ✓' with no name, no date, and no number is not a
> signature. It is a shrug."*
> — a **journal reviewer**, again, on what an attestation is worth

Round 1 exposed the breaks; you fixed them overnight. Round 2 is the **sign-off**.
Your partner re-runs your fixed package and, if your headline number comes back,
they **attest** to it — in their own words, dated and specific. The signature is
the whole point of the milestone: reproducibility claimed by you is a rumor;
reproducibility attested by someone else is evidence.

The attestation has a genre, and it is a strict one. Compare:

**An honest sign-off:** *"Priya Nair reproduced my headline number — the
18-point office-hours gap — from my package alone on Dec 2, 2026.
Restart-and-run-all returned 61% and 43% exactly. One residual: Figure 2 needed
a seed she found in my decision log but not in the cell; now fixed in cell 12."*
Named, dated, numbered, and honest about the one thing that did not come out
clean.

**A cosmetic non-sign-off:** *"Reproduced ✓."* No name, no date, no number, no
residual. The rubric reads this as an exchange that never really happened.

And the residuals principle that makes all of this safe to be honest about: if
some part does **not** reproduce, you log exactly what, why as far as you can
tell, and what a fix would need. **A documented failure is science; a hidden one
is misconduct.** The audit weighs the honesty of your residuals log as heavily
as a clean pass — so there is never a reason to fake the pass.

> **A question that often comes up here:** *"What if my partner's reproduction
> gives a slightly different number — is that a failure?"* It depends on *why*,
> and naming the why is the work. A different number because a seed was missing
> is a real residual (fix the seed). A different number because your partner's
> pandas rounds display differently is a cosmetic difference (note it and move
> on). The residuals log exists precisely to tell those two apart in writing —
> which is a skill worth more than a clean pass.

---

### 📝 Practice — sign-off or shrug?

Three partner responses to a completed reproduction. Sort each into **honest
attestation**, **honest residuals log**, or **cosmetic shrug** — and for the
shrug, say the one thing that would fix it:

- **A.** "Ran it Dec 2; got your 3.4-point difference on the nose. Signed, J. Osei."
- **B.** "Looks good to me 👍"
- **C.** "Re-ran clean except the map in cell 9 threw a font warning and the
  county figure came out 0.2 lower — I think it's a rounding path, not a data
  problem. Flagging it. — M. Alvarez, Dec 2."


### YOUR ANSWER HERE:

**A:** &nbsp; **B:** &nbsp; **C:**

**What would fix the shrug:**

---

## 4. The Research Brief: One Page, One Reader, One Figure

> *"I read forty of these a month. I will give yours ninety seconds, standing,
> between meetings. If I have to hunt for how sure you are, I will assume you
> are sure — and act on the cleanest version of your finding, which is usually
> the one you did not mean."*
> — a **policy stakeholder** (a dean who funds pilots), describing your real reader

A poster is read by people who came to look. An abstract is read by people
deciding whether to look. A **research brief** is read by a **decision-maker who
will never see your appendix** and has about ninety seconds. This is the reader
M21 is built for, and writing for them is a distinct discipline: a brief is not
a poster at 40%, and it is not an abstract with a chart. It is one page carrying
exactly four things, plus one figure:

- **The question** — in plain language a non-specialist acts on.
- **The finding** — what you found, **at its real boundary**.
- **The uncertainty — on the page, not in the appendix** — the one thing the
  reader most needs to know about how firm this is.
- **The implication** — the "so what" for *their* decision.
- **One figure** — honest (scale, base, missing cases), legible at a glance.

The hard discipline is that compression must never become **un-verification.**
A brief is allowed to say *less* than your dossier; it is never allowed to say
*more*. The failure this milestone is built to catch is quiet: a caveated,
disciplined finding from your ledger sliding back into a clean, confident,
caveat-free sentence — because the clean version reads better. If a sentence
upgrades the claim, **the ledger must be upgraded first, with evidence.**
Otherwise the caveat stays on the page.

You draft each of the four blocks straight from your **claim ledger** — the
finding block from your headline rows, the uncertainty block from their boundary
and sensitivity fields. Nothing on the page may exceed what a ledger row
licenses.

---

### 🛠️ Hands-On: one honest figure

A brief gets exactly one figure, so it has to be honest and legible at a glance.
The cell below draws the running example — an illustrative 18-point office-hours
gap in a 320-person pilot (61% vs 43%) — the way a brief should: bars from a
**zero baseline**, on the **full 0–100 scale**, labeled. Run it, then read the
self-check that follows: the honesty is in the axis, and the axis is asserted.

In [ ]:
# One honest figure for the brief — the office-hours gap, drawn from zero.
# Illustrative numbers, stated transparently: 61% vs 43% in a 320-person pilot.
groups = ["First-generation", "Continuing-generation"]
skip_rate = [61, 43]                        # % who reported skipping office hours
gap = skip_rate[0] - skip_rate[1]           # the headline: an 18-point gap

fig, ax = plt.subplots()
bars = ax.bar(groups, skip_rate, color=["#4C72B0", "#B0B0B0"])
ax.set_ylim(0, 100)                         # honest scale: full 0–100, no truncation
ax.set_ylabel("Reported office-hours skipping (%)")
ax.set_title(f"An {gap}-point gap in a 320-person pilot (illustrative)")
ax.bar_label(bars, fmt="%d%%")
plt.tight_layout()
plt.show()

print("Headline gap:", gap, "percentage points")

In [ ]:
# Self-check — the honesty of a bar chart lives in its axis.
assert gap == 18, "the illustrative gap should be 18 points (61 - 43)"
assert ax.get_ylim() == (0.0, 100.0), "axis was truncated — a cut y-axis overstates the gap"
print("✓ Self-check passed: 18-point gap, axis honest from 0 to 100.")

### 🔍 Reading the Evidence

In the cell below, answer two things about the figure you just drew. First: if
someone re-drew these same two numbers with the y-axis starting at 40 instead of
0, what would change about the *impression* — and why is the impression itself a
claim the brief is responsible for? Second: this figure carries a finding but
not its **uncertainty**. Write the one sentence of uncertainty you would place
next to it on the page, so the dean reads it in their ninety seconds and not in
an appendix they will never open.

### YOUR ANSWER HERE:

**What a y-axis from 40 would do to the impression (and why the impression is a claim):**

**The one uncertainty sentence I would put beside this figure:**

---

### 📝 Practice — compress without losing the caveat

Here is a disciplined, three-sentence finding straight from a ledger:

> *"In my pilot of 320 first-year respondents, first-generation respondents
> reported skipping office hours 61% of the time. Continuing-generation
> respondents reported 43%. This is an 18-point gap, but it is descriptive and
> limited to these 320 respondents; whether it holds in the full first-year
> population is not something this pilot can establish."*

In the cell below, compress it to **one sentence** for the brief — shorter, but
carrying the number, the sample boundary, **and** the uncertainty. Then write
the tempting **one-sentence version that drops the caveat**, and label it for
what it is.

> 💡 **Gemini Prompt:** "Here is my one-sentence brief finding, and the disciplined
> three-sentence version it came from: [paste both]. As a skeptical reviewer, tell
> me whether my compression quietly dropped the **sample boundary** or the
> **uncertainty** — and flag any word that upgrades a descriptive pilot into a
> claim about the whole population."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check each flag against your ledger row — the compressed sentence may say
>   *less* than the row, never *more*; you decide what the row licenses.
> - [ ] If a rewrite tightens the sentence, confirm the number, the boundary, and
>   the caveat all still ride along before you use it.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### YOUR ANSWER HERE:

**My one-sentence brief version (number + boundary + uncertainty intact):**

**The caveat-dropping version (label it: what does it quietly overclaim?):**

---

### 🎟️ Claim Ticket

**Claim Ticket #40** — in the cell below, write your project's **finding
sentence**, compressed for the brief, with the caveat intact. Read it aloud
before you write it down: if it survives being spoken to a stranger without a
follow-up "well, but…", it is ready.

### YOUR ANSWER HERE:

**My finding sentence, brief-compressed, caveat intact:**

---

---

*(Meeting M41 picks up here.)*

## 5. The Brief Workshop: Every Sentence Traces to a Row

> *"Show me the sentence, and I will ask you for the row. If you cannot point to
> where in your ledger a sentence comes from, that sentence is not evidence —
> it is decoration, and decoration is where overclaiming hides."*
> — a **thesis advisor**, teaching the trace-back habit

M21 is due tonight, and today it goes under everyone's eyes. The protocol is a
**table read**: all five briefs are read by everyone, and each reader leaves two
margin notes on each brief — the **sharpest sentence** and the **weakest
claim**. Then you revise. The discipline the workshop enforces is
**ledger-traceable writing**: *every sentence on the page traces to a row in
your claim ledger.* When a reader flags your weakest claim, the fix is not
cosmetic wordsmithing — it is letting the ledger discipline the sentence. If the
sentence wants to say more than its row licenses, either the ledger earns the
upgrade with evidence, or the sentence comes down to the row.

> **A question that often comes up here:** *"What if my sharpest sentence and my
> weakest claim are the same sentence?"* Then you have found the most important
> sentence in your brief, and you should be a little thrilled. A sentence that
> lands hard *and* reaches past its evidence is the overclaim in its most
> seductive form — the one a reader will believe *because* it is well written.
> Rewrite it so it lands just as hard at the boundary the ledger actually
> supports. That rewrite is the whole skill.

---

### 📝 Practice — trace it, or catch that you can't

Below is one sentence lifted from a draft brief, and the ledger row it claims to
rest on. Decide whether the sentence **traces** to the row, or **exceeds** it —
and if it exceeds it, write the sentence the row actually licenses.

> **Brief sentence:** *"First-generation undergraduates nationwide are 40% less
> likely to use academic support, so the college should redirect advising funds
> to them."*
>
> **Ledger row:** *Claim — in the 320-person pilot, first-generation respondents
> reported skipping office hours 18 points more often (61% vs 43%). Evidence —
> nb17 cell 14. Verification — recomputed by hand; partner-reproduced Dec 2.
> Boundary — descriptive, these 320 respondents only. Survived sensitivity? —
> held at 15 points under alternative binning.*

> 💡 **Gemini Prompt:** "Here is a sentence from my research brief and the single
> claim-ledger row it claims to rest on: [paste both]. As a skeptical reviewer,
> name every place the sentence claims more than the row licenses — a bigger
> number, a wider population, or a causal or policy leap — and rewrite it so it
> sits exactly on the row."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Confirm each over-reach it names is real by re-reading your actual ledger
>   row — the row, not Gemini, is the authority on what you may claim.
> - [ ] If the sentence wants to say more than the row, the fix is to upgrade the
>   **ledger** with evidence first — never to talk the reviewer out of the catch.
> - [ ] Log this use in your AI ledger: tool, task, verification.


### YOUR ANSWER HERE:

**Does the sentence trace to the row, or exceed it? Where exactly:**

**The sentence the row actually licenses:**

---

## 6. Staging the Evidence Defense

**Guiding question:** *what will the Evidence Defense actually ask of you — and why is preparing your weakest rows first the strongest way to get ready?*

The brief is your project's honest ambassador on paper. Monday and Wednesday you
defend it **out loud**. Here is the format, so nothing about it surprises you:

- **A 10-minute defense** in which you present three things — your **claims**
  (what you found, at its boundary), your **choices** (why this approach, this
  data, this measurement, this analysis), and your **verification** (how you
  know each headline result is real).
- **A 5-minute cross-examination** in which everyone in the room, and Professor
  Moreira, may open your claim ledger **to any row**, point at **any decision**,
  and ask how you verified **any result**.

Your **defense slots are drawn today** — some of you defend **Monday, Dec 7**,
the rest **Wednesday, Dec 9**. On both days, when you are not defending, you are
a cross-examiner, and that role is graded too. The defense is not a debate you
win by never yielding; it is a demonstration that you know exactly how far your
evidence reaches — which means the smartest preparation is not rehearsing your
strengths. It is **finding your three weakest rows and preparing them first.**

> **A question that often comes up here:** *"Isn't it a bad idea to go in having
> thought hardest about my weakest claims?"* It is the only good idea. The
> cross-examination will find them whether or not you did — the only question is
> whether you meet them with a prepared, honest boundary ("that row is
> descriptive; here is exactly what it can and cannot say") or with an improvised
> overclaim under pressure. The prepared concession is the strongest move in the
> room. The panicked bluff is the worst. You choose which one you bring by what
> you prep tonight.

---

### 🎯 Project Transfer — prep your three weakest rows

In the cell below, open your claim ledger and pick the **three rows you would
least like a cross-examiner to probe.** For each, write the honest boundary
sentence you will say when they do — the "here is exactly what this row can and
cannot establish" that turns a feared question into a prepared answer.

### YOUR ANSWER HERE:

**Weakest row 1 — the row, and my prepared boundary sentence:**

**Weakest row 2 — the row, and my prepared boundary sentence:**

**Weakest row 3 — the row, and my prepared boundary sentence:**

---

### 🎟️ Claim Ticket

**Claim Ticket #41** — in the cell below, name the single ledger row you least
want probed on your defense day, and write your **first sentence** about it — the
one you would open with if a cross-examiner pointed straight at it. One row, one
sentence.

### YOUR ANSWER HERE:

**The row I least want probed + my first sentence about it:**

---

## 7. Wrap-Up

Across three meetings you turned a finding into something that can *survive
you*. You audited a package the way a stranger would, watched your own work
break under a partner's cold hands, and fixed what broke; you earned a **signed
attestation** — reproducibility exercised by another human, not asserted — and
learned that an honest residuals log is worth as much as a clean pass. Then you
compressed fourteen weeks onto **one page** for a reader with power and no time,
and held the line that compression is never permission to un-verify. Finally you
staged the defense by doing the brave thing: preparing your weakest rows first.

> **"A result you cannot hand to a stranger is a rumor; a claim you cannot trace
> to a row is decoration. The package makes it runnable, the brief makes it
> readable, and the ledger makes both of them honest."**

Monday, nb19 opens the **Evidence Defense** — the moment the whole semester
becomes personal. You will stand behind your claims, your choices, and your
verification for ten minutes, and the room will cross-examine any of it for
five. Bring your dossier draft and your ledger; the rows you prepped tonight are
the ones that will be tested first. This is not a document to submit — it is a
defense to give.

---

## 8. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 21 'Planning' | reproducibility as a planned property of a design | package-anatomy checklist | extended (concept-level, Colab-scale honors projects)*
- *RDSS ch. 22 'Realization' | realizing a design; what breaks between plan and execution | the five package sins + restart-and-run-all test | extended*
- *RDSS ch. 23 'Integration' (opening) | pulling the pieces into one honest whole | the one-page brief as the dossier's ambassador | adapted*
- *fresh (brief P8) | the partner-exchange sign-off protocol, the residuals-honesty principle, and the ledger-traceable brief | — | newly-constructed-from-source-concept*

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 21–23. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).

*(No dataset is loaded in this notebook — every example is constructed
transparently in-cell, so there is no external data to attribute.)*

---

<center>

Thank you!

</center>